# BigMart Modeling and Final Prediction

This notebook handles the full modeling stage after preparation.

The workflow is simple: load the prepared training data, split only the training data, compare a few baseline regression models, lightly tune the best one, retrain the final model on the full prepared training set, and use the prepared test set only at the end for final predictions.

## 1. Imports

Only the libraries needed for modeling and final prediction are used here.

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR

pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

## 2. Load the Prepared Training Data

This notebook starts from the prepared training file created in notebook 1.

In [2]:
prepared_train = pd.read_csv("prepared_train.csv")

print("Prepared train shape:", prepared_train.shape)
prepared_train.head()

Prepared train shape: (8523, 16)


,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,Item_Category,Outlet_Age,Item_Visibility_Mean,Visibility_Ratio
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380,Food,14,0.017387,0.922960
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228,Drinks,4,0.019219,1.003057
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700,Food,14,0.020145,0.831990
3,FDX07,19.20,Regular,0.015274,Fruits and Vegetables,182.0950,OUT010,1998,Small,Tier 3,Grocery Store,732.3800,Food,15,0.020365,0.750000
4,NCD19,8.93,Non-Edible,0.008082,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052,Non-Consumable,26,0.012123,0.666667


## 3. Feature Setup and Train-Validation Split

Only the training data is split here.
The test data is not used for model comparison or tuning.

In [3]:
target = "Item_Outlet_Sales"
drop_columns = ["Item_Identifier", "Outlet_Establishment_Year"]

X = prepared_train.drop(columns=drop_columns + [target])
y = prepared_train[target]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

X_train: (6818, 13)
X_valid: (1705, 13)
y_train: (6818,)
y_valid: (1705,)


In [4]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train_prepared = preprocessor.fit_transform(X_train)
X_valid_prepared = preprocessor.transform(X_valid)

print("Prepared training shape:", X_train_prepared.shape)
print("Prepared validation shape:", X_valid_prepared.shape)

Prepared training shape: (6818, 48)
Prepared validation shape: (1705, 48)


## 4. Baseline Regression Models

A small set of baseline models is compared fairly on the same split.

In [5]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(random_state=RANDOM_STATE, n_estimators=200, n_jobs=1),
    "HistGradientBoostingRegressor": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
    "SVR": SVR()
}

results = []
trained_models = {}

for model_name, model in models.items():
    model.fit(X_train_prepared, y_train)
    predictions = model.predict(X_valid_prepared)

    results.append({
        "Model": model_name,
        "MAE": round(mean_absolute_error(y_valid, predictions), 2),
        "RMSE": round(root_mean_squared_error(y_valid, predictions), 2),
        "R2": round(r2_score(y_valid, predictions), 4)
    })

    trained_models[model_name] = model

results_df = pd.DataFrame(results).sort_values(by=["RMSE", "MAE"]).reset_index(drop=True)
results_df

,Model,MAE,RMSE,R2
0,HistGradientBoostingRegressor,729.11,1049.26,0.5949
1,Linear Regression,791.25,1068.99,0.5796
2,Random Forest Regressor,747.16,1073.22,0.5762
3,SVR,1179.21,1580.52,0.0809


## 5. Light Tuning of the Best Baseline Model

`HistGradientBoostingRegressor` is tuned with a small grid because it performed best as the baseline model.
The search is kept compact and RMSE is used as the main selection metric.

In [6]:
param_grid = {
    "learning_rate": [0.05, 0.1],
    "max_iter": [100, 200, 300],
    "max_leaf_nodes": [15, 31],
    "min_samples_leaf": [20, 40],
    "l2_regularization": [0.0, 0.1]
}

hgb_grid = GridSearchCV(
    estimator=HistGradientBoostingRegressor(random_state=RANDOM_STATE),
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=1,
    verbose=0
)

hgb_grid.fit(X_train_prepared, y_train)

best_params = hgb_grid.best_params_
print("Best parameters:")
best_params

Best parameters:


{'l2_regularization': 0.0,
 'learning_rate': 0.05,
 'max_iter': 100,
 'max_leaf_nodes': 15,
 'min_samples_leaf': 40}

In [7]:
baseline_hgb = trained_models["HistGradientBoostingRegressor"]
tuned_hgb = hgb_grid.best_estimator_

y_pred_baseline = baseline_hgb.predict(X_valid_prepared)
y_pred_tuned = tuned_hgb.predict(X_valid_prepared)

comparison_df = pd.DataFrame([
    {
        "Model": "Baseline HistGradientBoostingRegressor",
        "MAE": round(mean_absolute_error(y_valid, y_pred_baseline), 2),
        "RMSE": round(root_mean_squared_error(y_valid, y_pred_baseline), 2),
        "R2": round(r2_score(y_valid, y_pred_baseline), 4)
    },
    {
        "Model": "Tuned HistGradientBoostingRegressor",
        "MAE": round(mean_absolute_error(y_valid, y_pred_tuned), 2),
        "RMSE": round(root_mean_squared_error(y_valid, y_pred_tuned), 2),
        "R2": round(r2_score(y_valid, y_pred_tuned), 4)
    }
])

comparison_df

,Model,MAE,RMSE,R2
0,Baseline HistGradientBoostingRegressor,729.11,1049.26,0.5949
1,Tuned HistGradientBoostingRegressor,713.94,1018.95,0.6180


In [8]:
rmse_improvement = round(comparison_df.loc[0, "RMSE"] - comparison_df.loc[1, "RMSE"], 2)
mae_improvement = round(comparison_df.loc[0, "MAE"] - comparison_df.loc[1, "MAE"], 2)
r2_improvement = round(comparison_df.loc[1, "R2"] - comparison_df.loc[0, "R2"], 4)

print("RMSE improvement:", rmse_improvement)
print("MAE improvement:", mae_improvement)
print("R2 improvement:", r2_improvement)

RMSE improvement: 30.31
MAE improvement: 15.17
R2 improvement: 0.0231


## 6. Retrain the Final Model on the Full Prepared Training Data

Since the tuned model performs better, it is retrained on the full prepared training dataset.

In [9]:
X_full = prepared_train.drop(columns=drop_columns + [target])
y_full = prepared_train[target]

numeric_features_full = X_full.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features_full = X_full.select_dtypes(include=["object", "category"]).columns.tolist()

final_preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features_full),
    ("cat", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), categorical_features_full)
])

X_full_prepared = final_preprocessor.fit_transform(X_full)

final_model = HistGradientBoostingRegressor(random_state=RANDOM_STATE, **best_params)
final_model.fit(X_full_prepared, y_full)

print("Final model trained on full prepared training data.")

Final model trained on full prepared training data.


## 7. Use the Prepared Test Data Only at the End

The prepared test file is loaded only now, after model comparison and tuning are finished.

In [10]:
prepared_test = pd.read_csv("prepared_test.csv")
X_test = prepared_test.drop(columns=drop_columns)
X_test_prepared = final_preprocessor.transform(X_test)

print("Prepared test shape:", prepared_test.shape)
print("Final test matrix shape:", X_test_prepared.shape)

Prepared test shape: (5681, 15)
Final test matrix shape: (5681, 48)


## 8. Generate Final Predictions

The final model is used to predict `Item_Outlet_Sales` for the prepared test data.

In [11]:
final_predictions = final_model.predict(X_test_prepared)

prediction_df = pd.DataFrame({
    "Item_Identifier": prepared_test["Item_Identifier"],
    "Outlet_Identifier": prepared_test["Outlet_Identifier"],
    "Item_Outlet_Sales": final_predictions
})

prediction_df.head()

,Item_Identifier,Outlet_Identifier,Item_Outlet_Sales
0,FDW58,OUT049,1712.939463
1,FDW14,OUT017,1373.506572
2,NCN55,OUT010,585.538394
3,FDQ58,OUT017,2516.833454
4,FDY38,OUT027,6127.438365


In [12]:
prediction_df.to_csv("bigmart_final_predictions.csv", index=False)
print("Saved bigmart_final_predictions.csv")

Saved bigmart_final_predictions.csv
